<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_06-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 지능시스템 실습 2: BM25 직접 구현

이번 실습에서는 **BM25 검색 알고리즘을 직접 구현**합니다.

### 학습 목표
- TF(Term Frequency)가 무엇인지 이해한다.
- IDF(Inverse Document Frequency)가 무엇인지 이해한다.
- 문서 길이 정규화가 왜 필요한지 이해한다.
- BM25 점수를 직접 계산한다.
- 검색 질의에 따라 문서 순위가 어떻게 달라지는지 확인한다.

> 이 실습에서는 BM25의 원리를 쉽게 보기 위해 **간단한 공백 기반 토큰화**를 사용합니다.  
> 실제 한국어 검색 시스템에서는 형태소 분석기 또는 더 정교한 tokenizer를 사용할 수 있습니다.

## 1. 실습용 문서 준비

여러 개의 짧은 문서를 검색 대상으로 사용합니다.

In [1]:
documents = {
    "doc1": "RAG는 외부 문서를 검색하여 LLM 답변 생성에 활용하는 기술이다",
    "doc2": "BM25는 검색어와 문서의 단어 일치를 이용하는 키워드 기반 검색 방법이다",
    "doc3": "Dense Retrieval은 문장을 임베딩 벡터로 변환하여 의미가 비슷한 문서를 찾는다",
    "doc4": "RAG에서는 BM25와 Dense Retrieval을 함께 사용하는 Hybrid Retrieval이 많이 사용된다",
    "doc5": "Chunking은 긴 문서를 작은 단위로 나누어 검색하기 쉽게 만드는 과정이다"
}

for doc_id, text in documents.items():
    print(doc_id, ":", text)

doc1 : RAG는 외부 문서를 검색하여 LLM 답변 생성에 활용하는 기술이다
doc2 : BM25는 검색어와 문서의 단어 일치를 이용하는 키워드 기반 검색 방법이다
doc3 : Dense Retrieval은 문장을 임베딩 벡터로 변환하여 의미가 비슷한 문서를 찾는다
doc4 : RAG에서는 BM25와 Dense Retrieval을 함께 사용하는 Hybrid Retrieval이 많이 사용된다
doc5 : Chunking은 긴 문서를 작은 단위로 나누어 검색하기 쉽게 만드는 과정이다


## 2. 아주 간단한 한국어 토큰화 함수 만들기

BM25는 기본적으로 **단어가 문서에 몇 번 등장했는지**를 사용합니다.

따라서 먼저 문장을 단어 단위로 나눠야 합니다.

In [2]:
import re

def tokenize(text):
    """
    아주 간단한 tokenizer
    1. 영문은 소문자로 바꿉니다.
    2. 한글, 영문, 숫자만 남깁니다.
    3. 공백을 기준으로 나눕니다.
    """

    # 모두 소문자로 변환
    text = text.lower()

    # 한글/영문/숫자를 제외한 문장부호를 공백으로 바꿈
    text = re.sub(r"[^0-9a-zA-Z가-힣]+", " ", text)

    # 공백 기준으로 단어 분리
    tokens = text.split()

    return tokens


# 한 문장을 토큰으로 나눠 보기
sample = "BM25는 검색어와 문서의 관련성을 계산합니다."

print("원문:", sample)
print("토큰:", tokenize(sample))

원문: BM25는 검색어와 문서의 관련성을 계산합니다.
토큰: ['bm25는', '검색어와', '문서의', '관련성을', '계산합니다']


## 3. 모든 문서를 토큰화하기

In [3]:
# 각 문서를 token 목록으로 변환
tokenized_docs = {}

for doc_id, text in documents.items():
    tokenized_docs[doc_id] = tokenize(text)

# 결과 확인
for doc_id, tokens in tokenized_docs.items():
    print(doc_id, ":", tokens)

doc1 : ['rag는', '외부', '문서를', '검색하여', 'llm', '답변', '생성에', '활용하는', '기술이다']
doc2 : ['bm25는', '검색어와', '문서의', '단어', '일치를', '이용하는', '키워드', '기반', '검색', '방법이다']
doc3 : ['dense', 'retrieval은', '문장을', '임베딩', '벡터로', '변환하여', '의미가', '비슷한', '문서를', '찾는다']
doc4 : ['rag에서는', 'bm25와', 'dense', 'retrieval을', '함께', '사용하는', 'hybrid', 'retrieval이', '많이', '사용된다']
doc5 : ['chunking은', '긴', '문서를', '작은', '단위로', '나누어', '검색하기', '쉽게', '만드는', '과정이다']


## 4. TF(Term Frequency) 이해하기

TF는 **한 문서 안에서 어떤 단어가 몇 번 등장했는가**를 의미합니다.

예를 들어 `"검색"`이라는 단어가 한 문서에 여러 번 나오면  
그 문서는 검색과 관련 있을 가능성이 높다고 볼 수 있습니다.

In [4]:
from collections import Counter

# doc2의 단어 빈도 계산
tf_doc2 = Counter(tokenized_docs["doc2"])

print("doc2 단어 빈도:")
print(tf_doc2)

doc2 단어 빈도:
Counter({'bm25는': 1, '검색어와': 1, '문서의': 1, '단어': 1, '일치를': 1, '이용하는': 1, '키워드': 1, '기반': 1, '검색': 1, '방법이다': 1})


## 5. DF와 IDF 계산하기

### DF(Document Frequency)
어떤 단어가 **몇 개의 문서에 등장하는지** 계산합니다.

### IDF(Inverse Document Frequency)
여러 문서에 흔하게 등장하는 단어는 중요도를 낮추고,  
적은 문서에만 등장하는 희귀한 단어는 중요도를 높입니다.

이번 실습에서는 다음의 널리 쓰이는 BM25 IDF 형태를 사용합니다.

\[
IDF(t) = \log\left(1 + rac{N-df(t)+0.5}{df(t)+0.5}
ight)
\]

- `N`: 전체 문서 수
- `df(t)`: 단어 t가 등장한 문서 수

In [5]:
import math
from collections import defaultdict

# 전체 문서 수
N = len(tokenized_docs)

# 각 단어가 몇 개의 문서에 등장하는지 계산
df = defaultdict(int)

for tokens in tokenized_docs.values():
    # 같은 문서에서 같은 단어가 여러 번 나와도 DF에는 한 번만 계산
    unique_tokens = set(tokens)

    for token in unique_tokens:
        df[token] += 1


def compute_idf(term):
    """
    BM25에서 사용할 IDF 계산
    """
    term_df = df.get(term, 0)

    return math.log(
        1 + (N - term_df + 0.5) / (term_df + 0.5)
    )


# 몇 개 단어의 DF와 IDF 확인
terms = ["rag는", "검색", "bm25는", "문서를"]

for term in terms:
    print(
        f"{term:10s}",
        "DF =", df.get(term, 0),
        "IDF =", round(compute_idf(term), 4)
    )

rag는       DF = 1 IDF = 1.3863
검색         DF = 1 IDF = 1.3863
bm25는      DF = 1 IDF = 1.3863
문서를        DF = 3 IDF = 0.539


## 6. 문서 길이와 평균 문서 길이 계산하기

BM25는 긴 문서가 단순히 단어를 많이 포함한다는 이유로  
항상 높은 점수를 받지 않도록 **문서 길이를 보정**합니다.

In [6]:
# 각 문서의 길이 = token 개수
doc_lengths = {
    doc_id: len(tokens)
    for doc_id, tokens in tokenized_docs.items()
}

# 평균 문서 길이
avg_doc_length = sum(doc_lengths.values()) / len(doc_lengths)

print("문서 길이:")
for doc_id, length in doc_lengths.items():
    print(doc_id, ":", length)

print()
print("평균 문서 길이:", round(avg_doc_length, 2))

문서 길이:
doc1 : 9
doc2 : 10
doc3 : 10
doc4 : 10
doc5 : 10

평균 문서 길이: 9.8


## 7. BM25 점수 함수 구현하기

BM25의 핵심 점수는 다음과 같이 생각하면 됩니다.

1. 검색어가 문서에 많이 나오면 점수가 올라간다.
2. 하지만 너무 많이 반복되어도 증가 효과는 점점 줄어든다.
3. 흔한 단어보다 희귀한 단어를 더 중요하게 본다.
4. 긴 문서는 길이 때문에 유리하지 않도록 보정한다.

In [7]:
def bm25_score(query, doc_id, k1=1.5, b=0.75):
    """
    한 질의(query)와 한 문서(doc_id)의 BM25 점수를 계산합니다.
    """

    # 질의를 token으로 분리
    query_tokens = tokenize(query)

    # 대상 문서의 token 목록
    doc_tokens = tokenized_docs[doc_id]

    # 문서 안에서 각 단어가 몇 번 등장했는지 계산
    term_freq = Counter(doc_tokens)

    # 문서 길이
    doc_len = len(doc_tokens)

    score = 0.0

    # 질의의 각 단어에 대해 BM25 점수 계산
    for term in query_tokens:

        # 이 단어가 문서에서 몇 번 나오는가?
        f = term_freq.get(term, 0)

        # 문서에 단어가 없으면 점수에 기여하지 않음
        if f == 0:
            continue

        # 희귀한 단어일수록 높은 IDF
        idf = compute_idf(term)

        # BM25의 TF + 길이 정규화 부분
        numerator = f * (k1 + 1)

        denominator = (
            f
            + k1 * (
                1 - b
                + b * doc_len / avg_doc_length
            )
        )

        # 현재 단어의 점수를 전체 점수에 더함
        score += idf * (numerator / denominator)

    return score

## 8. 질의 하나로 전체 문서 검색하기

검색어를 넣으면 모든 문서의 BM25 점수를 계산한 뒤  
점수가 높은 순서대로 정렬합니다.

In [19]:
def bm25_search(query, k1=1.5, b=0.75):
    """
    모든 문서의 BM25 점수를 계산하여
    높은 점수 순서로 반환합니다.
    """

    results = []

    for doc_id in documents:
        score = bm25_score(query, doc_id, k1=k1, b=b)

        results.append({
            "문서": doc_id,
            "BM25 점수": score,
            "내용": documents[doc_id]
        })

    # BM25 점수가 높은 순서로 정렬
    results.sort(
        key=lambda x: x["BM25 점수"],
        reverse=True
    )

    return results


# 검색 실행
query = "LLM"

results = bm25_search(query)

print("질의:", query)
print()

for rank, result in enumerate(results, 1):
    print(
        rank,
        result["문서"],
        f"점수={result['BM25 점수']:.4f}"
    )
    print(" ", result["내용"])
    print()

질의: LLM

1 doc1 점수=1.4392
  RAG는 외부 문서를 검색하여 LLM 답변 생성에 활용하는 기술이다

2 doc2 점수=0.0000
  BM25는 검색어와 문서의 단어 일치를 이용하는 키워드 기반 검색 방법이다

3 doc3 점수=0.0000
  Dense Retrieval은 문장을 임베딩 벡터로 변환하여 의미가 비슷한 문서를 찾는다

4 doc4 점수=0.0000
  RAG에서는 BM25와 Dense Retrieval을 함께 사용하는 Hybrid Retrieval이 많이 사용된다

5 doc5 점수=0.0000
  Chunking은 긴 문서를 작은 단위로 나누어 검색하기 쉽게 만드는 과정이다



# 실습 정리

BM25는 크게 세 가지 아이디어를 사용합니다.

### 1. TF
질의 단어가 문서에 많이 등장할수록 관련성이 높다고 봅니다.

### 2. IDF
모든 문서에 흔한 단어보다 **희귀한 단어를 더 중요하게** 봅니다.

### 3. Length Normalization
긴 문서는 단어가 우연히 많이 등장할 수 있으므로 **문서 길이를 보정**합니다.

---

### 한 문장 정리

> **BM25는 질의 단어의 빈도, 희귀성, 문서 길이를 함께 고려하여 문서의 검색 관련성 점수를 계산합니다.**